# Unit Tests for `assessment_processor.py`

Unit-level and fixture-based validation for DOCX and HTML parser behavior.


In [1]:
from pathlib import Path
import re
import sys
import tempfile

from bs4 import BeautifulSoup
from docx import Document


def find_preprocessing_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        direct = candidate / "assessment_processor.py"
        nested = candidate / "preprocessing" / "assessment_processor.py"
        if direct.exists():
            return candidate
        if nested.exists():
            return nested.parent
    raise FileNotFoundError("Could not locate preprocessing/assessment_processor.py")


preprocessing_dir = find_preprocessing_dir(Path.cwd().resolve())
if str(preprocessing_dir) not in sys.path:
    sys.path.insert(0, str(preprocessing_dir))

from assessment_processor import AssessmentParser, parse_to_dict

parser = AssessmentParser()
TEST_DOCX_FOLDER = preprocessing_dir / "test_preprocessing" / "word assessment files"


def run_test(name, test_func):
    try:
        test_func()
    except Exception as exc:
        print(f"FAIL: {name}")
        print(f"{type(exc).__name__}: {exc}")
        raise
    else:
        print(f"PASS: {name}")


In [2]:
def cleanup(path: Path) -> None:
    try:
        path.unlink()
    except FileNotFoundError:
        pass


def make_temp_docx() -> Path:
    doc = Document()
    doc.add_heading("Main Section", level=1)
    doc.add_heading("Sub Section", level=2)

    paragraph = doc.add_paragraph()
    paragraph.add_run("Intro ")
    bold_run = paragraph.add_run("Bold")
    bold_run.bold = True
    paragraph.add_run(" ")
    italic_run = paragraph.add_run("Name")
    italic_run.italic = True

    table = doc.add_table(rows=2, cols=2)
    header_run = table.cell(0, 0).paragraphs[0].add_run("Header")
    header_run.bold = True
    table.cell(0, 1).text = "Value"
    table.cell(1, 0).text = "A"
    table.cell(1, 1).text = "B"

    temp = tempfile.NamedTemporaryFile(suffix=".docx", delete=False)
    temp.close()
    path = Path(temp.name)
    doc.save(path)
    return path


def make_temp_html(suffix=".html") -> Path:
    html = """
    <html>
      <body>
        <h1>HTML Section</h1>
        <h2>HTML Subsection</h2>
        <p>Plain <strong>bold text</strong> and <em>italic text</em></p>
        <ul><li>First item</li><li>Second item</li></ul>
        <table><tr><th>Header</th></tr><tr><td>Value</td></tr></table>
      </body>
    </html>
    """
    temp = tempfile.NamedTemporaryFile("w", suffix=suffix, delete=False, encoding="utf-8")
    temp.write(html)
    temp.close()
    return Path(temp.name)


def make_edge_case_docx() -> Path:
    doc = Document()
    doc.add_paragraph("Preface before heading")
    doc.add_heading("Orphan Subsection", level=2)
    doc.add_paragraph("Orphan body text")
    doc.add_heading("Main Section", level=1)
    doc.add_paragraph("Main body text")
    doc.add_heading("Repeated", level=2)
    doc.add_paragraph("First repeated text")
    doc.add_heading("Second Main", level=1)
    doc.add_heading("Repeated", level=2)
    doc.add_paragraph("Second repeated text")

    table = doc.add_table(rows=2, cols=2)
    table.cell(0, 0).text = "Key"
    table.cell(0, 1).text = "Value"
    table.cell(1, 0).text = "Empty right cell"

    temp = tempfile.NamedTemporaryFile(suffix=".docx", delete=False)
    temp.close()
    path = Path(temp.name)
    doc.save(path)
    return path


def walk_nodes(node: dict):
    yield node
    for child in node.get("children", []):
        yield from walk_nodes(child)


def all_blocks(node: dict):
    for current in walk_nodes(node):
        for block in current.get("blocks", []):
            yield block


def section_paths(data: dict):
    return {tuple(node.get("path", [])) for node in walk_nodes(data) if node.get("path")}


def find_node_by_path(data: dict, path):
    path = tuple(path)
    for node in walk_nodes(data):
        if tuple(node.get("path", [])) == path:
            return node
    raise AssertionError(f"Missing section path: {path}")


def strip_rich_tags(text: str) -> str:
    return re.sub(r"</?(?:b|i|sup|sub)>", "", text or "")


def assert_output_invariants(data: dict) -> None:
    required_node_keys = {"title", "level", "path", "blocks", "children"}
    valid_block_types = {"paragraph", "list", "table", "style"}

    assert isinstance(data.get("comments"), list)
    for node in walk_nodes(data):
        assert required_node_keys <= set(node)
        assert isinstance(node["path"], list)
        assert isinstance(node["blocks"], list)
        assert isinstance(node["children"], list)
        if node["path"]:
            assert node["title"] == node["path"][-1]

        for child in node["children"]:
            if node["path"]:
                assert child["path"][:-1] == node["path"]
            assert child["level"] in {1, 2}

        for block in node["blocks"]:
            assert block.get("type") in valid_block_types
            if block.get("type") == "paragraph":
                assert isinstance(block.get("text"), str) and block["text"]
            elif block.get("type") == "list":
                assert isinstance(block.get("items"), list)
                assert all(isinstance(item.get("text"), str) and item["text"] for item in block["items"])
            elif block.get("type") == "table":
                assert isinstance(block.get("rows"), list)
                assert all(isinstance(row, list) for row in block["rows"])
            elif block.get("type") == "style":
                assert isinstance(block.get("data"), dict)

    required_comment_keys = {
        "id", "author", "date", "text", "anchor_heading_path",
        "anchor_text", "anchor_text_rich", "anchor_context", "anchor_context_rich",
    }
    for comment in data.get("comments", []):
        assert required_comment_keys <= set(comment)


def assert_rich_text_matches_plain_text(data: dict) -> None:
    for block in all_blocks(data):
        if "text_rich" in block:
            assert strip_rich_tags(block["text_rich"]) == block.get("text", "")
        for item in block.get("items", []):
            if "text_rich" in item:
                assert strip_rich_tags(item["text_rich"]) == item.get("text", "")
        rows = block.get("rows", [])
        rows_rich = block.get("rows_rich", [])
        if rows_rich:
            assert len(rows_rich) == len(rows)
            for plain_row, rich_row in zip(rows, rows_rich):
                assert len(rich_row) == len(plain_row)
                assert [strip_rich_tags(cell) for cell in rich_row] == plain_row


## Test 1: Rich Text Wrapper

Expected nesting order for superscript, italic, and bold rich-text tags.


In [3]:
def test_wrap_rich_text():
    result = parser._wrap_rich_text("Name", bold=True, italic=True, vertical="superscript")
    assert result == "<b><i><sup>Name</sup></i></b>"


run_test("rich text wrapper", test_wrap_rich_text)

PASS: rich text wrapper


## Test 2: Paragraph Rich Text

DOCX paragraph rendering for subscript, superscript, bold, and italic text.


In [4]:
def test_paragraph_rich_text():
    doc = Document()
    paragraph = doc.add_paragraph()
    paragraph.add_run("CO")
    subscript_run = paragraph.add_run("2")
    subscript_run.font.subscript = True
    paragraph.add_run(" and km")
    superscript_run = paragraph.add_run("2")
    superscript_run.font.superscript = True
    bold_run = paragraph.add_run(" bold")
    bold_run.bold = True
    italic_run = paragraph.add_run(" italic")
    italic_run.italic = True

    result = parser._paragraph_text_with_scripts(paragraph)
    assert result == "CO<sub>2</sub> and km<sup>2</sup><b> bold</b><i> italic</i>"


run_test("paragraph rich text", test_paragraph_rich_text)

PASS: paragraph rich text


## Test 3: Style Bucket Merge

Ordered de-duplication of extracted style snippets.


In [5]:
def test_style_bucket_merge():
    target = {"bold": ["A"], "italic": []}
    source = {"bold": ["A", "B", ""], "italic": ["I", "I"]}
    parser._merge_style_bucket(target, source)

    assert target["bold"] == ["A", "B"]
    assert target["italic"] == ["I"]


run_test("style bucket merge", test_style_bucket_merge)

PASS: style bucket merge


## Test 4: Heading Detection

Mapping of Heading 1 and Heading 2 paragraph styles to parser heading levels.


In [6]:
def test_heading_detection():
    doc = Document()
    h1 = doc.add_heading("Main", level=1)
    h2 = doc.add_heading("Sub", level=2)
    normal = doc.add_paragraph("Body")

    assert parser._heading_level(h1) == 1
    assert parser._heading_level(h2) == 2
    assert parser._heading_level(normal) is None


run_test("heading detection", test_heading_detection)

PASS: heading detection


## Test 5: Table Extraction

Plain-text and rich-text extraction from DOCX tables.


In [7]:
def test_table_extraction():
    doc = Document()
    table = doc.add_table(rows=2, cols=2)
    header_run = table.cell(0, 0).paragraphs[0].add_run("Header")
    header_run.bold = True
    table.cell(0, 1).text = "Value"
    table.cell(1, 0).text = "A"
    table.cell(1, 1).text = "B"

    assert parser._table_to_rows(table) == [["Header", "Value"], ["A", "B"]]
    assert parser._table_to_rows_rich(table) == [["<b>Header</b>", "Value"], ["A", "B"]]


run_test("table extraction", test_table_extraction)

PASS: table extraction


## Test 6: Full DOCX Parsing

Parsing of a generated DOCX fixture into headings, paragraph blocks, table blocks, and style data.


In [8]:
def test_full_docx_parsing():
    path = make_temp_docx()
    try:
        data = parser.parse_file(str(path))
    finally:
        cleanup(path)

    assert data["title"] == path.stem
    assert data["children"][0]["title"] == "Main Section"
    subsection = data["children"][0]["children"][0]
    assert subsection["title"] == "Sub Section"
    assert subsection["blocks"][0]["text"] == "Intro Bold Name"
    assert subsection["blocks"][0]["text_rich"] == "Intro <b>Bold</b> <i>Name</i>"
    assert subsection["blocks"][1]["rows"][1] == ["A", "B"]
    assert subsection["blocks"][1]["rows_rich"][0][0] == "<b>Header</b>"
    assert "Bold" in subsection["blocks"][-1]["data"]["bold"]
    assert "Name" in subsection["blocks"][-1]["data"]["italic"]


run_test("full docx parsing", test_full_docx_parsing)

PASS: full docx parsing


## Test 7: DOCX Without Comments

Comment output for a DOCX file without Word comments.


In [9]:
def test_docx_without_comments():
    path = make_temp_docx()
    try:
        data = parser.parse_file(str(path))
    finally:
        cleanup(path)

    assert data["comments"] == []


run_test("docx without comments", test_docx_without_comments)

PASS: docx without comments


## Test 8: HTML Parsing

Parsing of a generated HTML fixture into headings, paragraph, list, table, and style blocks.


In [10]:
def test_html_parsing():
    path = make_temp_html()
    try:
        data = parser.parse_file(str(path))
    finally:
        cleanup(path)

    assert data["title"] == path.stem
    assert data["comments"] == []
    assert data["children"][0]["title"] == "HTML Section"
    subsection = data["children"][0]["children"][0]
    assert subsection["title"] == "HTML Subsection"
    assert subsection["blocks"][0] == {"type": "paragraph", "text": "Plain bold text and italic text"}
    assert subsection["blocks"][1]["type"] == "list"
    assert subsection["blocks"][2]["rows"] == [["Header"], ["Value"]]
    assert subsection["blocks"][-1]["type"] == "style"


run_test("html parsing", test_html_parsing)

PASS: html parsing


## Test 9: HTML Style Extraction

Extraction of tag-based, class-based, and inline-style bold/italic snippets from HTML.


In [11]:
def test_html_style_extraction():
    soup = BeautifulSoup(
        """
        <p>
          <b>Strong</b>
          <i>Emphasis</i>
          <span class="dataLabel">Label</span>
          <span style="font-weight:700">Heavy</span>
          <span style="font-style: italic">Lean</span>
        </p>
        """,
        "html.parser",
    )
    styles = parser._extract_styles_from_html_element(soup.p)

    assert "Strong" in styles["bold"]
    assert "Label" in styles["bold"]
    assert "Heavy" in styles["bold"]
    assert "Emphasis" in styles["italic"]
    assert "Lean" in styles["italic"]


run_test("html style extraction", test_html_style_extraction)

PASS: html style extraction


## Test 10: `.htm` File Support

Support for `.htm` files through the HTML parsing path.


In [12]:
def test_htm_extension_support():
    path = make_temp_html(suffix=".htm")
    try:
        data = parser.parse_file(str(path))
    finally:
        cleanup(path)

    assert data["children"][0]["title"] == "HTML Section"


run_test("htm extension support", test_htm_extension_support)

PASS: htm extension support


## Test 11: `parse_to_dict` Helper

Public helper parsing for a supported document path.


In [13]:
def test_parse_to_dict_helper():
    path = make_temp_html()
    try:
        data = parse_to_dict(str(path))
    finally:
        cleanup(path)

    assert data["children"][0]["title"] == "HTML Section"


run_test("parse_to_dict helper", test_parse_to_dict_helper)

PASS: parse_to_dict helper


## Test 12: Raw XML Run Rendering

Rendering of direct bold, italic, and superscript formatting from raw DOCX XML runs.


In [14]:
def test_xml_run_text_rendering():
    doc = Document()
    paragraph = doc.add_paragraph()
    run = paragraph.add_run("Raw")
    run.bold = True
    run.italic = True
    run.font.superscript = True

    result = parser._xml_run_text_with_scripts(run._r)
    assert result == "<b><i><sup>Raw</sup></i></b>"


run_test("xml run text rendering", test_xml_run_text_rendering)

PASS: xml run text rendering


## Test 13: Unsupported File Extension

Error handling for unsupported file types.


In [15]:
def test_unsupported_file_extension():
    temp = tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False, encoding="utf-8")
    temp.write("not supported")
    temp.close()
    path = Path(temp.name)

    try:
        try:
            parser.parse_file(str(path))
        except ValueError as exc:
            assert "Unsupported file type" in str(exc)
        else:
            raise AssertionError("Expected ValueError for unsupported file type")
    finally:
        cleanup(path)


run_test("unsupported file extension", test_unsupported_file_extension)

PASS: unsupported file extension


## Test 14: DOCX Edge Structure

Parsing of root-level text, an orphan Heading 2, repeated subsection names, and a table with a blank cell.


In [16]:
def test_black_box_docx_edge_structure():
    path = make_edge_case_docx()
    try:
        data = parser.parse_file(str(path))
    finally:
        cleanup(path)

    assert data["blocks"][0]["text"] == "Preface before heading"
    paths = section_paths(data)
    assert ("Orphan Subsection",) in paths
    assert ("Main Section", "Repeated") in paths
    assert ("Second Main", "Repeated") in paths

    orphan = find_node_by_path(data, ("Orphan Subsection",))
    first_repeated = find_node_by_path(data, ("Main Section", "Repeated"))
    second_repeated = find_node_by_path(data, ("Second Main", "Repeated"))

    assert orphan["blocks"][0]["text"] == "Orphan body text"
    assert first_repeated["blocks"][0]["text"] == "First repeated text"
    assert second_repeated["blocks"][0]["text"] == "Second repeated text"
    assert second_repeated["blocks"][1]["rows"] == [["Key", "Value"], ["Empty right cell", ""]]


run_test("docx edge structure", test_black_box_docx_edge_structure)


PASS: docx edge structure


## Test 15: Synthetic Output Invariants

Structural invariants and rich-text consistency for generated DOCX and HTML parser outputs.


In [17]:
def test_synthetic_output_invariants():
    docx_path = make_temp_docx()
    html_path = make_temp_html()
    try:
        docx_data = parser.parse_file(str(docx_path))
        html_data = parser.parse_file(str(html_path))
    finally:
        cleanup(docx_path)
        cleanup(html_path)

    for data in (docx_data, html_data):
        assert_output_invariants(data)
        assert_rich_text_matches_plain_text(data)


run_test("synthetic output invariants", test_synthetic_output_invariants)


PASS: synthetic output invariants


## Test 16: Real DOCX Fixture Regressions

Stable parser facts for representative assessment documents included in the test fixtures.


In [18]:
def test_real_docx_fixture_regressions():
    assert TEST_DOCX_FOLDER.exists()
    assert len(list(TEST_DOCX_FOLDER.glob("*.docx"))) >= 100

    expected = {
        "Acrocarpus_fraxinifolius_JP (2).docx": {"comments": 11, "min_paths": 25},
        "Gardenia truncata Craib_JP.docx": {"comments": 5, "min_paths": 28},
        "Microchirita hemratii_JP.docx": {"comments": 3, "min_paths": 33},
    }

    required_paths = {
        ("Red List Assessment",),
        ("Red List Assessment", "Assessment Information"),
        ("Distribution",),
        ("Distribution", "Geographic Range"),
        ("Threats",),
        ("Conservation",),
    }

    for filename, facts in expected.items():
        path = TEST_DOCX_FOLDER / filename
        data = parser.parse_file(str(path))
        paths = section_paths(data)

        assert data["title"] == path.stem
        assert len(data.get("comments", [])) == facts["comments"]
        assert len(paths) >= facts["min_paths"]
        assert required_paths <= paths
        assert_output_invariants(data)


run_test("real docx fixture regressions", test_real_docx_fixture_regressions)


PASS: real docx fixture regressions


## Test 17: Deterministic Parser Output

Deterministic JSON output when parsing the same fixture twice.


In [19]:
def test_parse_determinism():
    path = TEST_DOCX_FOLDER / "Acrocarpus_fraxinifolius_JP (2).docx"
    first = parser.parse_file(str(path))
    second = parser.parse_file(str(path))
    assert first == second


run_test("parse determinism", test_parse_determinism)


PASS: parse determinism
